# Lexel Stage 1: QLoRA SFT on Qwen2.5-1.5B-Instruct (Kaggle T4 / Unsloth Studio)

This notebook trains the Stage 1 Supervised Fine-Tuning (SFT) adapter for **Lexel** using Unsloth 4-bit QLoRA.

## Architecture & Invariants
- **Base Models**: `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` (Standard Tier, ~1.04 GB GGUF) or `unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit` (Lite Tier, ~469 MB GGUF).
- **LoRA Configuration**: `r=32`, `lora_alpha=64`, `lora_dropout=0`, targeting all 7 attention and MLP linear projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`) with `use_gradient_checkpointing="unsloth"`.
- **Completion-Only Loss & Sequence Packing**: Supervises assistant completions only (`train_on_responses_only`) with `max_seq_length=1536`, `lr=2e-4`, cosine schedule, `warmup_ratio=0.05`, `num_train_epochs=2`, and `optim="adamw_8bit"`.
- **6-Gate Data Crucible**: Trains exclusively on `lexel_sft_train.jsonl` and validates on `lexel_sft_val.jsonl` across all 13 Lexel surfaces and 7 tasks.

In [ ]:
# 1. Install Unsloth and TRL on Kaggle Free Tier (2x T4 16GB / P100)
# Skip this cell when running inside Unsloth Studio Desktop where packages are pre-installed.
import os
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install --no-deps unsloth

In [ ]:
# 2. Configuration & Model Tier Toggle
import hashlib
import json
import time
from datetime import UTC, datetime
from pathlib import Path

import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# Toggle between "lexel-1.5b" (Standard) and "lexel-0.5b" (Lite)
MODEL_TIER = os.environ.get("LEXEL_MODEL_TIER", "lexel-1.5b")
MODEL_MAP = {
    "lexel-1.5b": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    "lexel-0.5b": "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
}
BASE_MODEL_ID = MODEL_MAP[MODEL_TIER]
MAX_SEQ_LENGTH = 1536
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

DATA_DIR = Path(os.environ.get("LEXEL_DATA_DIR", "../datasets"))
TRAIN_FILE = DATA_DIR / "lexel_sft_train.jsonl"
VAL_FILE = DATA_DIR / "lexel_sft_val.jsonl"
OUTPUT_DIR = Path(os.environ.get("LEXEL_SFT_OUT", f"./outputs/{MODEL_TIER}-sft-adapter"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Selected Tier: {MODEL_TIER} -> {BASE_MODEL_ID}")
print(f"Train dataset: {TRAIN_FILE}")
print(f"Val dataset:   {VAL_FILE}")

In [ ]:
# 3. Load 4-bit Base Model & Attach LoRA Adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

In [ ]:
# 4. Load Crucible-Verified Datasets & Apply Qwen-2.5 ChatML Formatting
dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VAL_FILE),
    },
)

def format_chat_prompts(batch):
    texts = [
        tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        for msgs in batch["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(format_chat_prompts, batched=True)
print(f"Train rows: {len(dataset['train'])}, Validation rows: {len(dataset['validation'])}")
print("\nSample formatted ChatML prompt:\n", dataset["train"][0]["text"][:600])

In [ ]:
# 5. Configure SFTTrainer with Completion-Only Masking & Sequence Packing
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=sft_config,
)

# Apply strict completion-only loss masking (-100 on all system/user prompt tokens)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# 6. Execute Stage 1 Training & Save SFT Adapter + run_manifest.json
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
train_stats = trainer.train()
elapsed_sec = time.perf_counter() - t0

peak_vram_gb = (
    round(torch.cuda.max_memory_allocated() / (1024**3), 3)
    if torch.cuda.is_available()
    else 0.0
)

adapter_dir = OUTPUT_DIR / "adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

train_sha256 = hashlib.sha256(TRAIN_FILE.read_bytes()).hexdigest()
val_sha256 = hashlib.sha256(VAL_FILE.read_bytes()).hexdigest()

run_manifest = {
    "stage": "stage1_sft",
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "model_tier": MODEL_TIER,
    "base_model_id": BASE_MODEL_ID,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora": {
        "r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "use_gradient_checkpointing": "unsloth",
        "adapter_path": str(adapter_dir),
    },
    "hyperparameters": {
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "num_train_epochs": 2,
        "optim": "adamw_8bit",
        "packing": True,
        "train_on_responses_only": True,
    },
    "datasets": {
        "train_file": str(TRAIN_FILE),
        "train_rows": len(dataset["train"]),
        "train_sha256": train_sha256,
        "val_file": str(VAL_FILE),
        "val_rows": len(dataset["validation"]),
        "val_sha256": val_sha256,
    },
    "telemetry": {
        "global_step": int(trainer.state.global_step),
        "training_loss": round(float(train_stats.training_loss), 6),
        "wall_time_sec": round(elapsed_sec, 2),
        "peak_vram_gb": peak_vram_gb,
    },
}

manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2) + "\n", encoding="utf-8")
print("Saved Stage 1 SFT adapter and run_manifest.json to:", OUTPUT_DIR)